# Notebook 07: Grover's Algorithm and Its Geometry

Grover's search algorithm finds a marked item in an unsorted database of $N$
items using only $O(\sqrt{N})$ queries -- a quadratic speedup over the classical
$O(N)$.  But the *mechanism* is not brute-force parallel checking.  It is
**constructive interference** via geometric rotation.

In this notebook we will:
1. Build the oracle and diffusion operator
2. Run Grover's algorithm and track the target probability
3. Visualize the probability trajectory over iterations
4. Reveal the 2D rotation geometry underlying the algorithm
5. Compute optimal iteration counts for various problem sizes

**Key message:** Grover's algorithm amplifies the target state through
interference and geometry, NOT through parallel brute-force search.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
from quantum_demo.linalg import ket, is_unitary
from quantum_demo.states import equal_superposition, amplitudes_to_probabilities, pretty_basis_labels
from quantum_demo.gates import phase_oracle, diffusion_operator, apply_gate
from quantum_demo.grover import (
    grover_iteration, grover_run, grover_optimal_iterations,
    target_probability_trajectory, reduced_grover_plane_coordinates
)
from quantum_demo.viz.static_plots import plot_probabilities, plot_real_amplitudes
from quantum_demo.viz.grover_plots import plot_grover_trajectory, plot_grover_plane

## 1. The two ingredients: Oracle and Diffusion

Grover's algorithm uses exactly two operations, applied alternately:

1. **Oracle** $O_f$: Flips the sign of the target state's amplitude. All other
   amplitudes are unchanged. This is a diagonal matrix with $-1$ at the target
   index and $+1$ everywhere else.

2. **Diffusion operator** $D = 2|s\rangle\langle s| - I$: Reflects all amplitudes
   about their mean, where $|s\rangle$ is the equal superposition state.

In [ ]:
# Working example: N=8 states (3 qubits), target at index 5
N = 8
target = 5

oracle = phase_oracle(target, N)
diffusion = diffusion_operator(N)

print(f"Oracle for target={target} in {N}-dimensional space")
print(f"Oracle diagonal: {np.diag(oracle).real}")
print(f"Is unitary? {is_unitary(oracle)}")
print(f"\nDiffusion operator shape: {diffusion.shape}")
print(f"Is unitary? {is_unitary(diffusion)}")

## 2. Running Grover's algorithm step by step

Each Grover iteration applies the oracle followed by the diffusion operator.
Let's watch the state evolve through several iterations.

In [ ]:
# Compute optimal number of iterations
k_opt = grover_optimal_iterations(N)
print(f"For N={N}: optimal iterations = {k_opt}")
print(f"  (pi/4) * sqrt({N}) = {np.pi/4 * np.sqrt(N):.2f}")

# Run Grover for a few extra iterations to see the oscillation
n_iters = k_opt + 4
trajectory = grover_run(N, target, n_iters)

print(f"\nTrajectory contains {len(trajectory)} states (initial + {n_iters} iterations)")

In [ ]:
labels = pretty_basis_labels(3)

# Show the state at each iteration
for i, state in enumerate(trajectory):
    p_target = amplitudes_to_probabilities(state)[target]
    print(f"Iteration {i}: P(target={labels[target]}) = {p_target:.4f}")

Notice the probability of the target state *rises* with each iteration, peaks,
and then *falls* again.  Grover's algorithm is like a pendulum -- you must
stop at the right moment.

## 3. Probability trajectory plot

Let's use `target_probability_trajectory` and `plot_grover_trajectory` for a
clean visualization.

In [ ]:
# Probability of target at each step
probs = target_probability_trajectory(N, target, n_iters)

fig = plot_grover_trajectory(
    probs,
    title=f"Grover Search: P(target) vs Iteration (N={N}, target={labels[target]})"
)
fig

The target probability oscillates sinusoidally.  At the optimal iteration
count, it reaches nearly 1.0.  If you keep iterating, it drops back down --
the algorithm "overshoots."

## 4. Amplitude snapshots

Let's visualize the full amplitude vector at a few key moments.

In [ ]:
import matplotlib.pyplot as plt

key_steps = [0, 1, k_opt]
fig, axes = plt.subplots(1, len(key_steps), figsize=(5 * len(key_steps), 3.5))

for ax, step in zip(axes, key_steps):
    state = trajectory[step]
    p_t = amplitudes_to_probabilities(state)[target]
    plot_real_amplitudes(
        state, labels=labels,
        title=f"Iteration {step}\nP(target)={p_t:.3f}",
        ax=ax
    )

fig.suptitle("Grover's Algorithm: Amplitude Evolution", fontsize=13, y=1.05)
fig.tight_layout()
fig

Watch how the target amplitude (bar at position `101`) grows taller with each
iteration while the non-target amplitudes shrink.  This is constructive
interference on the target and destructive interference on everything else.

## 5. The 2D rotation geometry

The deepest insight into Grover's algorithm comes from recognizing that the
entire evolution takes place in a **2-dimensional plane** spanned by:

- $|t\rangle$ -- the target state
- $|s_\perp\rangle$ -- the equal superposition of all *non-target* states

The initial state $|s\rangle$ (equal superposition over all states) is a vector
in this plane that is close to $|s_\perp\rangle$ (since there are $N-1$
non-target states but only 1 target).

Each Grover iteration rotates the state vector toward $|t\rangle$ by a fixed
angle $2\theta$, where $\sin\theta = 1/\sqrt{N}$.

Let's project the trajectory onto this plane.

In [ ]:
# Compute 2D coordinates for each state in the trajectory
coords = [reduced_grover_plane_coordinates(state, target) for state in trajectory]

print("2D coordinates (x = |target> component, y = |s_perp> component):")
for i, (x, y) in enumerate(coords):
    print(f"  Iteration {i}: ({x:.4f}, {y:.4f})  |  norm = {np.sqrt(x**2 + y**2):.4f}")

In [ ]:
fig = plot_grover_plane(
    coords,
    title=f"Grover Geometry: Rotation in 2D Plane (N={N})"
)
fig

### Reading the geometry plot

- Each arrow is the state vector after one iteration.
- The state starts near the $|s_\perp\rangle$ axis (bottom-right area) and
  rotates toward the $|t\rangle$ axis (left side).
- All state vectors lie on the unit circle (norm = 1).
- The rotation angle per iteration is constant: $2\theta$ where
  $\sin\theta = 1/\sqrt{N}$.
- At the optimal iteration count, the state vector points nearly along
  $|t\rangle$ -- meaning the target probability is nearly 1.

This is why Grover's algorithm works: it is a *rotation*, not a search.
The oracle and diffusion operator together implement a fixed-angle rotation
in a 2D plane, and after $\sim \frac{\pi}{4}\sqrt{N}$ rotations, the state
aligns with the target.

## 6. Optimal iterations for various problem sizes

The optimal number of Grover iterations scales as $\frac{\pi}{4}\sqrt{N}$.
Let's see this scaling in action.

In [ ]:
print(f"{'N':>8} | {'dim (2^n)':>10} | {'Optimal iters':>14} | {'Classical queries':>17} | {'Speedup':>8}")
print("-" * 70)

for n_qubits in range(2, 21):
    dim = 2 ** n_qubits
    k_opt = grover_optimal_iterations(dim)
    classical = dim  # worst case: check every item
    speedup = classical / max(k_opt, 1)
    print(f"{n_qubits:>8} | {dim:>10,} | {k_opt:>14,} | {classical:>17,} | {speedup:>8.0f}x")

For 20 qubits ($N = 1{,}048{,}576$ items), Grover needs only about 804
iterations instead of a million classical checks -- a **1,000x speedup**.

The speedup grows as $\sqrt{N}$, which is a quadratic advantage.  While
this is not exponential, it is provably optimal for unstructured search.

## 7. A larger example: 5 qubits (32 states)

Let's run Grover on a slightly larger problem to see the same pattern at
higher dimension.

In [ ]:
N_large = 32  # 5 qubits
target_large = 13
k_opt_large = grover_optimal_iterations(N_large)

print(f"N={N_large}, target={target_large}, optimal iterations={k_opt_large}")

# Run and plot trajectory
probs_large = target_probability_trajectory(N_large, target_large, k_opt_large + 6)
fig = plot_grover_trajectory(
    probs_large,
    title=f"Grover Search: N={N_large}, target={target_large}"
)
fig

In [ ]:
# 2D geometry for the larger example
traj_large = grover_run(N_large, target_large, k_opt_large + 2)
coords_large = [reduced_grover_plane_coordinates(s, target_large) for s in traj_large]

fig = plot_grover_plane(
    coords_large,
    title=f"Grover Geometry: N={N_large}"
)
fig

With more states, the initial angle $\theta$ is smaller (the state starts
closer to $|s_\perp\rangle$), so more iterations are needed to rotate to
$|t\rangle$.  But the rotation is still clean and geometric.

## Key takeaways

- Grover's algorithm consists of alternating **oracle** (phase flip) and **diffusion** (reflection about mean) operations.
- The target probability **oscillates** sinusoidally; you must measure at the right iteration.
- The optimal number of iterations is $\sim \frac{\pi}{4}\sqrt{N}$, giving a **quadratic speedup** over classical search.
- The algorithm is fundamentally a **2D rotation** in the plane spanned by $|t\rangle$ and $|s_\perp\rangle$.
- Grover amplifies via **interference and geometry**, not parallel brute-force checking.
- The speedup is provably optimal for unstructured search -- no quantum algorithm can do better.

**Next notebook:** We will generate publication-quality figures from all the demos covered so far.